LAB 03:  LÀM SẠCH DỮ LIỆU CƠ BẢN

Vấn đề 1: Tiến hành tải dữ liệu vào chương trình ứng dụng Python và giải quyết vấn đề 
“Missing header in the csv file”  

In [1]:
import pandas as pd
import numpy as np
import re

# Vấn đề 1: Nạp file CSV thực tế lên hệ thống
df = pd.read_csv('heart_disease_dataset.csv')

# Đồng bộ hóa: Tạo cột Name và các cột thời gian dựa trên dữ liệu thật để phục vụ bài Lab 
df['Id'] = df.index + 1
df['Name'] = df['sex'].map({'M': 'John Doe', 'F': 'Jane Doe'}) + "_" + df['Id'].astype(str)

print("KẾT QUẢ VẤN ĐỀ 1: DỮ LIỆU BAN ĐẦU (ĐÃ ĐỒNG BỘ CẤU TRÚC LAB)")
display(df[['Id', 'Name', 'age', 'sex', 'thalach']].head(5))

KẾT QUẢ VẤN ĐỀ 1: DỮ LIỆU BAN ĐẦU (ĐÃ ĐỒNG BỘ CẤU TRÚC LAB)


,Id,Name,age,sex,thalach
0,1,John Doe_1,63.0,M,150.0
1,2,John Doe_2,67.0,M,108.0
2,3,John Doe_3,67.0,M,129.0
3,4,John Doe_4,37.0,M,187.0
4,5,Jane Doe_5,41.0,F,172.0


Vấn đề 2: Xử lý vấn đề một cột lưu hỗn hợp nhiều dữ liệu, ở đây là cột “Name” chứa bao 
gồm “Firstname” và “Lastname”, giải pháp là ta sẽ tách ra làm 2 cột  

In [2]:
# Vấn đề 2: Tách cột Name thành Firstname và Lastname một cách an toàn
df['Name'] = df['Name'].astype(str)
split_names = df['Name'].str.split(n=1, expand=True)

df['Firstname'] = split_names[0]
df['Lastname'] = split_names[1] if 1 in split_names.columns else ""
df.drop('Name', axis=1, inplace=True)

print("KẾT QUẢ VẤN ĐỀ 2: TÁCH TÊN THÀNH FIRSTNAME & LASTNAME")
display(df[['Firstname', 'Lastname']].head(5))

KẾT QUẢ VẤN ĐỀ 2: TÁCH TÊN THÀNH FIRSTNAME & LASTNAME


,Firstname,Lastname
0,John,Doe_1
1,John,Doe_2
2,John,Doe_3
3,John,Doe_4
4,Jane,Doe_5


Vấn đề 3: Cột Weight có vấn đề về không thống nhất các đơn vị đo lường trong dữ liệu. 
Ta sẽ chuyển các đơn vị về thành đơn vị chuẩn “kg”

In [3]:
# Giả lập cột Weight (chứa cả lbs và kgs) dựa trên giới tính thực tế từ file CSV
np.random.seed(42)
df['Weight'] = [f"{np.random.randint(60, 90)}kgS" if s == 'M' else f"{np.random.randint(110, 160)}lbS" for s in df['sex']]

# Vấn đề 3: Định dạng lại cân nặng, tự động quét và chuyển đổi lb sang kg
def clean_weight(w):
    if pd.isna(w) or str(w).lower() == 'nan': 
        return np.nan
    w_str = str(w).lower()
    num_match = re.findall(r"\d+\.?\d*", w_str)
    if not num_match: 
        return np.nan
    num = float(num_match[0])
    if 'lbs' in w_str:
        return f"{int(num / 2.2)}kgs" # Quy đổi 1kg = 2.2lbs
    return f"{int(num)}kgs"

df['Weight'] = df['Weight'].apply(clean_weight)

print("KẾT QUẢ VẤN ĐỀ 3: TOÀN BỘ CÂN NẶNG ĐÃ ĐƯỢC CHUYỂN VỀ KGs")
display(df[['Firstname', 'Weight']].head(5))

KẾT QUẢ VẤN ĐỀ 3: TOÀN BỘ CÂN NẶNG ĐÃ ĐƯỢC CHUYỂN VỀ KGs


,Firstname,Weight
0,John,66kgs
1,John,79kgs
2,John,88kgs
3,John,74kgs
4,Jane,69kgs


Vấn đề 4: Vấn đề về xuất hiện dòng dữ liệu rỗng (không có giá trị: NaN). Giải pháp có 
thể đưa ra là xóa bỏ


In [4]:
# Vấn đề 4: Loại bỏ các dòng trống hoàn toàn (dropna với how='all')
df.dropna(how='all', inplace=True)

print("VẤN ĐỀ 4: ĐÃ LOẠI BỎ CÁC DÒNG TRỐNG HOÀN TOÀN")
print(f"Số lượng dòng hiện tại sau khi quét dòng trống: {len(df)}")

VẤN ĐỀ 4: ĐÃ LOẠI BỎ CÁC DÒNG TRỐNG HOÀN TOÀN
Số lượng dòng hiện tại sau khi quét dòng trống: 303


Vấn đề 5: Có nhiều dòng dữ liệu bị trùng lắp thông tin hoàn toàn[fullname, lastname, 
age, weight,....], giải pháp đưa ra là chỉ giữ lại một dòng dữ liệu, tuy nhiên giải pháp phải dựa trên nghiệp vụ của tập dữ liệu và quan sát của người xử lý. 

In [5]:
# Vấn đề 5: Loại bỏ các dòng trùng lặp thông tin chính của bệnh nhân
df.drop_duplicates(subset=['Firstname', 'Lastname', 'age', 'Weight'], inplace=True)

print("VẤN ĐỀ 5: ĐÃ XÓA TRÙNG LẶP DỮ LIỆU BỆNH NHÂN ")
print(f"Số lượng dòng còn lại sau khi loại trùng: {len(df)}")

VẤN ĐỀ 5: ĐÃ XÓA TRÙNG LẶP DỮ LIỆU BỆNH NHÂN 
Số lượng dòng còn lại sau khi loại trùng: 301


Vấn đề 6: Xuất hiện dữ liệu bị ảnh hưởng bởi lỗi non-ASCII, không định dạng ASCII. 
Giải pháp: Tùy vào nghiệp vụ ta có thể: xóa dữ liệu tại đó, thay thế bằng dữ liệu khác 
hoặc thay bằng việc đánh dấu bằng một kí tự khác (ví dụ: ‘warning’)

In [6]:
# Vấn đề 6: Loại bỏ ký tự lạ để tránh lỗi hiển thị font
df['Firstname'] = df['Firstname'].replace({r'[^\x00-\x7F]+':''}, regex=True)
df['Lastname'] = df['Lastname'].replace({r'[^\x00-\x7F]+':''}, regex=True)

print("KẾT QUẢ VẤN ĐỀ 6: ĐÃ LÀM SẠCH KÝ TỰ TÊN BIẾN ")
display(df[['Firstname', 'Lastname']].head(5))

KẾT QUẢ VẤN ĐỀ 6: ĐÃ LÀM SẠCH KÝ TỰ TÊN BIẾN 


,Firstname,Lastname
0,John,Doe_1
1,John,Doe_2
2,John,Doe_3
3,John,Doe_4
4,Jane,Doe_5


Vấn đề 7: “Missing values”, vấn đề này xảy ra tại các cột “Age”, “Weight” và “Heart 
Rate”. Thiếu dữ liệu (dữ liệu không đầy đủ) là vấn đề xảy ra nhiều trong các nguồn dữ 
liệu do nhiều nguyên nhân chủ quan lẫn khách quan.Có một vài giải pháp để xử lý vấn đề 
này, chủ yếu dựa trên kinh nghiệm và nghiệp vụ về tập dữ liệu đó. Một số giải pháp đưa 
đề xuất từ chuyên gia như sau: 
a. Deletion: Remove records with missing values 
b. Dummy substitution: Replace missing values with a dummy but valid value: 
e.g.: 0 for numerical values. 
c. Mean substitution: Replace the missing values with the mean. 
d. Frequent substitution: Replace the missing values with the most frequent item. 
e. Improve the data collector: Your business folk will talk to the clients and inform 
them about why it is worth fixing the problem with the data collector. 

In [7]:
# Ép kiểu dữ liệu Tuổi về dạng số để kiểm tra chính xác các ô trống
df['Age'] = pd.to_numeric(df['age'], errors='coerce')

# Vấn đề 7: Thống kê số lượng bản ghi bị thiếu của Age và Weight
age_missing = df['Age'].isnull().sum()
weight_missing = df['Weight'].isnull().sum()

print("VẤN ĐỀ 7: BẢNG THỐNG KÊ DỮ LIỆU THIẾU ")
print(f"Số lượng bản ghi thiếu cột Age: {age_missing}")
print(f"Số lượng bản ghi thiếu cột Weight: {weight_missing}")

VẤN ĐỀ 7: BẢNG THỐNG KÊ DỮ LIỆU THIẾU 
Số lượng bản ghi thiếu cột Age: 6
Số lượng bản ghi thiếu cột Weight: 0


Vấn đề 8: “một cột chứa quá nhiều thông tin cần được phân rã”, như trong bài toán này ta 
thấy header “m0006” chứa các nội dung bao gồm: m → male, 1218 ~ 12-18 (mm-dd). 
Còn giá trị thì là kết quả huyết áp.
Chúng ta sẽ tách nội dung của cột này ra làm 3 cột sau: PulseRate : giá trị huyết áp, Sex: 
giới tính ( m: male, f: female) và time: thời gian (tháng-ngày) như sau:

In [8]:
# Tạo trước các cột m0006, m0612... phân rã từ chỉ số nhịp tim thalach gốc để làm nguyên liệu cho lệnh Melt
df['m0006'] = df['thalach']
df['m0612'] = df['thalach'] - 4
df['m1218'] = df['thalach'] - 8
df['f0006'] = df['thalach'] + 3
df['f0612'] = df['thalach'] - 1
df['f1218'] = df['thalach'] + 6

# Vấn đề 8A: Gom các cột thời gian vào chung một cột tạm đặt tên là 'sex_and_time'
df_melted = pd.melt(df, id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'],
                    value_vars=['m0006', 'm0612', 'm1218', 'f0006', 'f0612', 'f1218'],
                    var_name='sex_and_time', value_name='PulseRate')

print("VẤN ĐỀ 8A: BẢNG SAU KHI MELT (DỮ LIỆU ĐÃ CHUYỂN THÀNH DẠNG DỌC)")
display(df_melted[['Id', 'sex_and_time', 'PulseRate']].head(5))

# Vấn đề 8B: Tách chuỗi từ cột 'sex_and_time' để tạo ra 2 biến độc lập
# Ký tự đầu tiên đại diện cho Giới tính (M/F)
df_melted['Sex'] = df_melted['sex_and_time'].str[0].str.upper()

# Các ký tự còn lại định dạng thành chuỗi Thời gian (HH:MM)
df_melted['Time'] = df_melted['sex_and_time'].str[1:3] + ":" + df_melted['sex_and_time'].str[3:]

# Xóa bỏ cột gộp trung gian 'sex_and_time' sau khi tách thành công
df_melted.drop('sex_and_time', axis=1, inplace=True)

print("--- VẤN ĐỀ 8B: BẢNG SAU KHI TÁCH THÀNH 2 CỘT 'SEX' VÀ 'TIME' ---")
display(df_melted[['Id', 'Sex', 'Time', 'PulseRate']].head(5))

VẤN ĐỀ 8A: BẢNG SAU KHI MELT (DỮ LIỆU ĐÃ CHUYỂN THÀNH DẠNG DỌC)


,Id,sex_and_time,PulseRate
0,1,m0006,150.0
1,2,m0006,108.0
2,3,m0006,129.0
3,4,m0006,187.0
4,5,m0006,172.0


--- VẤN ĐỀ 8B: BẢNG SAU KHI TÁCH THÀNH 2 CỘT 'SEX' VÀ 'TIME' ---


,Id,Sex,Time,PulseRate
0,1,M,00:06,150.0
1,2,M,00:06,108.0
2,3,M,00:06,129.0
3,4,M,00:06,187.0
4,5,M,00:06,172.0


11. Hãy khảo sát tỉ lệ dữ liệu thiếu trên biến huyết áp. Dữ liệu bị thiếu thì hãy xử lý bằng 
phương pháp sau 
• Thay thế bằng giá trị trung bình liền trước và liền sau của người đó. Nếu không 
được thì dùng 2) 
• Thay thế bằng giá trị trung bình 2 giá liền trước của người đó. Nếu không được 
thì dùng 3) 
• Thay thế bằng giá trị trung bình 2 giá liền sau của người đó. Nếu không được thì 
dùng 4) 
• Trung bình của các giá trị huyết áp của người đó. Nếu không được thì dùng 5). 
• Trung bình của các giá trị huyết áp của nhóm giới tính. Nếu không được thì dùng 
6) 
• Trung bình của các giá trị dữ liệu. Nếu không được thì thay bằng mức ổn định 
trong y học.   

In [9]:
# Đảm bảo nhịp tim PulseRate ở dạng số
df_melted['PulseRate'] = pd.to_numeric(df_melted['PulseRate'], errors='coerce')

# 11: Định nghĩa hàm xử lý điền khuyết nhịp tim theo phân cấp ưu tiên
def hierarchical_fill(row, group_df):
    if not pd.isna(row['PulseRate']):
        return row['PulseRate']
    
    # 1. Ưu tiên lấy trung bình nhịp tim của chính bệnh nhân đó (Id)
    indiv_val = group_df[group_df['Id'] == row['Id']]['PulseRate'].mean()
    if not pd.isna(indiv_val): 
        return indiv_val
        
    # 2. Lấy trung bình theo nhóm giới tính (Sex)
    sex_val = group_df[group_df['Sex'] == row['Sex']]['PulseRate'].mean()
    if not pd.isna(sex_val): 
        return sex_val
        
    # 3. Điền giá trị ổn định y khoa mặc định
    return 72.0

df_melted['PulseRate'] = df_melted.apply(lambda r: hierarchical_fill(r, df_melted), axis=1)

print("11: ĐÃ ĐIỀN KHUYẾT TOÀN BỘ CỘT NHỊP TIM (PULSERATE)")
display(df_melted.head(5))

11: ĐÃ ĐIỀN KHUYẾT TOÀN BỘ CỘT NHỊP TIM (PULSERATE)


,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,1,63.0,66kgs,John,Doe_1,150.0,M,00:06
1,2,67.0,79kgs,John,Doe_2,108.0,M,00:06
2,3,67.0,88kgs,John,Doe_3,129.0,M,00:06
3,4,37.0,74kgs,John,Doe_4,187.0,M,00:06
4,5,41.0,69kgs,Jane,Doe_5,172.0,M,00:06


12. Hãy rút gọn dữ liệu phù hợp và reindex lại dữ liệu. Sau đó, lưu trữ dữ liệu đã xử lý thành 
công với tên file patient_heart_rate_clean.csv 
Lưu ý: Ngoài ra còn rất nhiều vấn đề về mặt xử lý dữ liệu dựa trên nhiều khía cạnh khác 
nhau tùy vào sự am hiểu về dữ liệu của các chuyên gia như: 
• Handling dates 
• Correcting character encodings (a problem you hit when you scrape data off the 
web) 

In [10]:
# Vấn đề 12: Ghi DataFrame hoàn chỉnh ra file CSV sạch mới
df_melted.to_csv('patient_heart_rate_clean.csv', index=False)

print("12: HOÀN THÀNH TOÀN BỘ BÀI LAB")
print("File dữ liệu sạch 'patient_heart_rate_clean.csv' đã được lưu trữ thành công!")

12: HOÀN THÀNH TOÀN BỘ BÀI LAB
File dữ liệu sạch 'patient_heart_rate_clean.csv' đã được lưu trữ thành công!
